In [1]:
# Importy i ustawienie katalogu głównego projektu
import sys, os, re
import pandas as pd
import numpy as np
from pathlib import Path

# Przejdź do katalogu głównego projektu tylko raz — bezpieczne przy wielokrotnym uruchomieniu
if os.path.basename(os.getcwd()) != "analiza_sprzedazy_w_sklepie":
    os.chdir("../..")

sys.path.insert(0, os.getcwd())
import utils.tools as tools


In [2]:
# Suma kontrolna wejścia — ten sam hash u obu osób = identyczne dane
nazwa_pliku = "fact_inka_calendar.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   fact_inka_calendar.parquet   dd831a813d662099be81de7050891487f7e95808cf0c2aad19404e7a4484ef6e


# Flagi towarowe
**CzyWazone — towar sprzedawany na wagę**

In [3]:
# Wczytanie kalendarza (TowId × Data + ruchy magazynowe) do df_flagi
df_flagi = pd.read_parquet(f"dane/interim/{nazwa_pliku}")

### Towary wazone 

In [4]:
# ============================================================
# SOFT FILTER — Kryterium 2: Towary ważone (per TowId)
# Krok 1: % transakcji z ułamkową ilością per TowId
# ============================================================

# LiczbaTransakcji = wszystkie wiersze towaru (też puste dni i ruchy), LiczbaUlamkowych = IloscPlus z ułamkiem
wazone_check = df_flagi.groupby('TowId').agg(
    LiczbaTransakcji=('IloscPlus', 'count'),
    LiczbaUlamkowych=('IloscPlus', lambda x: (x % 1 != 0).sum()),
).reset_index()

wazone_check['PctUlamkowych'] = (
    wazone_check['LiczbaUlamkowych'] / wazone_check['LiczbaTransakcji'] * 100
)

print(wazone_check['PctUlamkowych'].describe())

count    12481.000000
mean         0.675678
std          6.449838
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         97.207528
Name: PctUlamkowych, dtype: float64


In [5]:
# Wykrywanie wagi po nazwie: samodzielne 'KG' (bez liczby przed nim) = towar na wagę
def czy_nazwa_sugeruje_wage(nazwa):
    nazwa = str(nazwa).upper().strip()
    
    # Wyklucz przypadki "liczba+KG" (stała gramatura opakowania, np. "0,5KG", "1,65KG")
    bez_gramatury = re.sub(r'\d+[\.,]?\d*\s*KG\b', '', nazwa)
    
    # Szukamy samodzielnego "KG" jako jednostki sprzedaży (bez liczby bezpośrednio przed nim)
    return bool(re.search(r'\bKG\b', bez_gramatury))

# Szybki test kontrolny
testy = [
    "Chleb mieszany 0.6 kg PRECELEK",         # False - gramatura
    "PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL", # False - gramatura
    "MĄKA ZIEMNIACZANA 0,5KG",                # False - gramatura
    "SURÓWKA KG",                              # True - waga
    "BAKŁAŻAN KG POLSKA",                      # True - waga
    "SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK",      # True - waga
]
for t in testy:
    print(f"{czy_nazwa_sugeruje_wage(t)}: {t}")

False: Chleb mieszany 0.6 kg PRECELEK
False: PROSZEK PERSIL UNIVERSAL 1,65KG HENKEL
False: MĄKA ZIEMNIACZANA 0,5KG
True: SURÓWKA KG
True: BAKŁAŻAN KG POLSKA
True: SAŁATKA WIELKANOCNA KG GRZEŚKOWIAK


In [6]:
# Finalna flaga JestWazony per TowId: >= 50% ułamkowych ilości LUB nazwa z samodzielnym 'KG'
nazwy_do_sprawdzenia = df_flagi[['TowId', 'NazwaTow']].drop_duplicates(subset='TowId')
nazwy_do_sprawdzenia['NazwaSugerujeWage'] = nazwy_do_sprawdzenia['NazwaTow'].apply(czy_nazwa_sugeruje_wage)

wazone_check_pelne = wazone_check.merge(nazwy_do_sprawdzenia, on='TowId', how='left')

wazone_check_pelne['JestWazony'] = (
    (wazone_check_pelne['PctUlamkowych'] >= 50) |
    (wazone_check_pelne['NazwaSugerujeWage'])
)

lista_wazonych_towid = set(wazone_check_pelne[wazone_check_pelne['JestWazony']]['TowId'])
print(f"Ważonych TowId (finalne kryterium): {len(lista_wazonych_towid)}")

dodane_przez_nazwe = wazone_check_pelne[
    (wazone_check_pelne['NazwaSugerujeWage']) & 
    (wazone_check_pelne['PctUlamkowych'] < 50)
]
print(f"\nDodatkowo złapane przez samodzielne 'KG' (bez liczby): {len(dodane_przez_nazwe)}")
print(dodane_przez_nazwe[['TowId', 'NazwaTow', 'PctUlamkowych']].sort_values('NazwaTow'))

Ważonych TowId (finalne kryterium): 193

Dodatkowo złapane przez samodzielne 'KG' (bez liczby): 118
       TowId                           NazwaTow  PctUlamkowych
4988   51262                          ANANAS KG      11.165644
5390   52141                 BAKŁAŻAN KG POLSKA       9.375000
3564   25652      BRZUSZKI Z ŁOSOSIA WĘDZONE KG       8.280255
4382   49625                          BURAKI KG      11.061552
9166   77611      CIASTKA MARKIZY KG  DR GERARD      10.152284
...      ...                                ...            ...
2735   16441              SER SALAMI MLEKPOL KG      46.389646
2748   16479   SER Z ORZECHEM WŁOSKIM KG ŁOWICZ      47.916667
11332  80088                         SURÓWKA KG      11.764706
5853   53068  USZKA Z GRZYBAMI KG KUCHNIA POLKI       2.726146
8870   67234               ZIEMNIAKI SŁODKIE KG      34.860558

[118 rows x 3 columns]


In [7]:
# Przypisanie JestWazony do wierszy + kontrola, z jakich asortymentów pochodzą ważone towary
df_flagi['JestWazony'] = df_flagi['TowId'].isin(lista_wazonych_towid)

print(df_flagi['JestWazony'].value_counts())

nazwy_wazonych = (
    df_flagi[df_flagi['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    9118366
True      367506
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
RYBY WĘDZONE WAGA /ryby           3
WARZYWA KISZONE                   3
PIECZYWO                          2
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
MIĘSO, WĘDLINY I GARMAŻERKA       1
WIELKANOCNE                       1
***PRZECENY                       1
ŚWIĄTECZNE                        1
CIASTKA                           1
Name: count, dtype: int64


In [8]:
# DUPLIKAT poprzedniej komórki (ten sam kod i wynik) — można usunąć
df_flagi['JestWazony'] = df_flagi['TowId'].isin(lista_wazonych_towid)

print(df_flagi['JestWazony'].value_counts())

nazwy_wazonych = (
    df_flagi[df_flagi['TowId'].isin(lista_wazonych_towid)]
    [['TowId', 'NazwaTow', 'NazwaAsort']]
    .drop_duplicates(subset='TowId')
)
print(nazwy_wazonych['NazwaAsort'].value_counts())

JestWazony
False    9118366
True      367506
Name: count, dtype: int64
NazwaAsort
CUKIERKI WAGA                    42
SERY WAGA /nabiał                33
WARZYWA                          31
OWOCE                            19
CUKIERKI                         16
WĘDLINY WAGA                     10
CIASTKA WAGA                      8
PIEROGI KOPYTKA KROKIETY INNE     6
WĘDLINY PACZKOWANE                4
RYBY WĘDZONE WAGA /ryby           3
WARZYWA KISZONE                   3
PIECZYWO                          2
RYBY MROŻONE WAGA                 2
PRZETWORY RYBNE /ryby             2
MARKA WŁASNA SPAR                 2
KAWY                              2
SURÓWKI I SAŁATKI                 2
MIĘSO DROBIOWE WAGA               1
MIĘSO, WĘDLINY I GARMAŻERKA       1
WIELKANOCNE                       1
***PRZECENY                       1
ŚWIĄTECZNE                        1
CIASTKA                           1
Name: count, dtype: int64


## CzyMartwy — brak sprzedaży w ostatnich N dniach 

In [9]:
# ============================================================
# CzyMartwy — z uwzględnieniem sezonowości (próg per-TowId)
# Liczone wyłącznie na SPRZEDAŻY (TypDok == 21, DokId != -1).
# Ruchy PZ / BO / REM itd. nie "ożywiają" towaru.
# ============================================================

df_flagi = df_flagi.sort_values(['TowId', 'Data']).reset_index(drop=True)

# Maska: realna sprzedaż (puste dni mają DokId = -1, inne ruchy mają TypDok != 21)
jest_sprzedaz = (df_flagi['DokId'] != -1) & (df_flagi['TypDok'] == 21)

# 1) Data ostatniej sprzedaży (forward-fill w obrębie TowId)
df_flagi['DataOstatniejSprzedazy'] = df_flagi['Data'].where(jest_sprzedaz)
df_flagi['DataOstatniejSprzedazy'] = (
    df_flagi.groupby('TowId')['DataOstatniejSprzedazy'].ffill()
)

df_flagi['DniOdSprzedazy'] = (
    df_flagi['Data'] - df_flagi['DataOstatniejSprzedazy']
).dt.days

# 2) Historyczne przerwy między KOLEJNYMI dniami sprzedaży, per TowId
dni_sprzedazy = (
    df_flagi.loc[jest_sprzedaz, ['TowId', 'Data']]
    .drop_duplicates()
    .sort_values(['TowId', 'Data'])
)
dni_sprzedazy['PoprzedniaData'] = dni_sprzedazy.groupby('TowId')['Data'].shift(1)
dni_sprzedazy['Przerwa'] = (dni_sprzedazy['Data'] - dni_sprzedazy['PoprzedniaData']).dt.days

# 3) Własny "rekord" normalnej przerwy każdego TowId (95 percentyl — odporny na pojedynczy wyjątek)
prog_indywidualny = (
    dni_sprzedazy.groupby('TowId')['Przerwa']
    .quantile(0.95)
    .rename('PrzerwaHistoryczna95pct')
)

# Data ostatniej sprzedaży per TowId (do flagi "na koniec danych")
ostatnia_sprzedaz = dni_sprzedazy.groupby('TowId')['Data'].max()

# 4) Próg per TowId = margines nad własną historią, z podłogą dla towarów z krótką historią.
#    Towary z jednym dniem sprzedaży nie mają przerwy (NaN) -> dostają samą podłogę.
N_MINIMUM = 550          # podłoga progu — dla towarów bez wyraźnej sezonowości
MARGINES = 1.5           # ile razy dłuższa niż zwykle musi być przerwa, żeby uznać za "martwy"

df_flagi = df_flagi.merge(prog_indywidualny, on='TowId', how='left')
df_flagi['PrognMartwy'] = np.maximum(
    N_MINIMUM,
    df_flagi['PrzerwaHistoryczna95pct'].fillna(0) * MARGINES
)

# 5) Flaga dynamiczna: czy W TYM DNIU przerwa od ostatniej sprzedaży przekroczyła próg.
#    Wiersze przed pierwszą sprzedażą towaru (DniOdSprzedazy = NaN) dostają False.
#    Uwaga: kalendarz jest przycięty do "ostatnia sprzedaż + 14 dni", więc dla towarów, które
#    przestały się sprzedawać, ta flaga zapala się tylko na ruchach (PZ/ST/...) po długiej przerwie.
df_flagi['CzyMartwyWTymDniu'] = df_flagi['DniOdSprzedazy'] >= df_flagi['PrognMartwy']

# 6) Flaga "na koniec danych": czy towar jest martwy na ostatni dzień danych
#    (dni od ostatniej sprzedaży do końca danych >= próg towaru). Ta sama wartość w każdym wierszu TowId.
data_max = df_flagi['Data'].max()
dni_do_konca = (data_max - ostatnia_sprzedaz).dt.days
prog_towid = df_flagi.drop_duplicates('TowId').set_index('TowId')['PrognMartwy']
martwy_na_koniec = dni_do_konca >= prog_towid.reindex(dni_do_konca.index)
df_flagi['CzyMartwyNaKoniecDanych'] = (
    df_flagi['TowId'].map(martwy_na_koniec).fillna(False).astype(bool)
)

print("CzyMartwyWTymDniu (wiersze):")
print(df_flagi['CzyMartwyWTymDniu'].value_counts())
print("\nCzyMartwyNaKoniecDanych (unikalne TowId):")
print(df_flagi.drop_duplicates('TowId')['CzyMartwyNaKoniecDanych'].value_counts())
print(f"\nPróg indywidualny — opis:\n{df_flagi['PrognMartwy'].describe()}")

CzyMartwyWTymDniu (wiersze):
CzyMartwyWTymDniu
False    9469168
True       16704
Name: count, dtype: int64

CzyMartwyNaKoniecDanych (unikalne TowId):
CzyMartwyNaKoniecDanych
False    9383
True     3098
Name: count, dtype: int64

Próg indywidualny — opis:
count    9.485872e+06
mean     5.531368e+02
std      3.602579e+01
min      5.500000e+02
25%      5.500000e+02
50%      5.500000e+02
75%      5.500000e+02
max      1.420500e+03
Name: PrognMartwy, dtype: float64


In [10]:

# ============================================================
# CzySezonowy — informacyjna flaga (niezależna od CzyMartwy)
# ============================================================

# Zobaczmy najpierw rozkład, żeby próg nie był z sufitu
print(df_flagi.drop_duplicates('TowId')['PrzerwaHistoryczna95pct'].describe())
print(df_flagi.drop_duplicates('TowId')['PrzerwaHistoryczna95pct'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count    11846.000000
mean        44.195003
std         80.849101
min          1.000000
25%          7.950000
50%         16.000000
75%         39.937500
max        947.000000
Name: PrzerwaHistoryczna95pct, dtype: float64
0.50     16.0000
0.75     39.9375
0.90    109.8000
0.95    199.0000
0.99    400.6025
Name: PrzerwaHistoryczna95pct, dtype: float64


In [11]:
# CzySezonowy = 1, gdy typowa przerwa w sprzedaży (95 percentyl) wynosi >= PROG_SEZONOWY dni
PROG_SEZONOWY = 100  # dni — zgodny z wcześniejszym prog_zagrozony, tuż przy 90. percentylu (109.8)

towid_sezonowe = (
    df_flagi.drop_duplicates('TowId')
    .loc[lambda d: d['PrzerwaHistoryczna95pct'] >= PROG_SEZONOWY, 'TowId']
)

df_flagi['CzySezonowy'] = df_flagi['TowId'].isin(towid_sezonowe)

print(df_flagi.drop_duplicates('TowId')['CzySezonowy'].value_counts())

CzySezonowy
False    11184
True      1297
Name: count, dtype: int64


In [ ]:
# Flagi jako int8 (0/1): mniejszy plik i bezproblemowy zapis do parquet; podgląd jednego wiersza
df_flagi['CzySezonowy'] = df_flagi['CzySezonowy'].astype('int8')
df_flagi['CzyMartwyWTymDniu'] = df_flagi['CzyMartwyWTymDniu'].astype('int8')
df_flagi['CzyMartwyNaKoniecDanych'] = df_flagi['CzyMartwyNaKoniecDanych'].astype('int8')


,TowId,Data,DokId,Kolejnosc,NrPozycji,TypPoz,IloscPlus,IloscMinus,CenaPoRab,Wartosc,...,TypRuchu,CzyNiechciane,JestWazony,DataOstatniejSprzedazy,DniOdSprzedazy,PrzerwaHistoryczna95pct,PrognMartwy,CzyMartwyWTymDniu,CzyMartwyNaKoniecDanych,CzySezonowy
4208981,23717,2024-12-20,1876640,12.0,12.0,4.0,1.0,0.0,3.1333,3.1333,...,sprzedaz,0,False,2024-12-20,0.0,13.6,550.0,0,0,0


In [13]:
# Zapis wyniku (kalendarz + wszystkie flagi) do dane/interim
nazwa_pliku = "fact_inka_final.parquet"
df_flagi.to_parquet(
    f"dane/interim/{nazwa_pliku}",
    compression='zstd',
    index=False
)
print(f"Zapisano: {df_flagi.shape}")

Zapisano: (9485872, 38)


In [14]:
# Suma kontrolna wyniku — przelicz po każdej zmianie kodu

moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")
# Uwaga: hash poniżej pochodzi ze starej wersji (37 kolumn) — po uruchomieniu zaktualizuj
#Mój hash (posortowane):   fact_inka_final.parquet   bfb8dc26980b014aff480c0d60d82a68af4dcbc818434c463db37ef62ba2573f

Mój hash (posortowane):   fact_inka_final.parquet   9b255e47616b01ade38ff34994625ddd177e811d38b9cf67603066667506da3a
